# Task 4: Prioritize Drug Target Genes

Using the perturbation results from Task 3, rank the 12 ALS target genes by how
effectively their perturbation shifts sALS cells toward the healthy (PN) state.

## Scoring methodology

Two complementary metrics are computed in Task 3 for each of the 24 perturbations
(12 genes × knockup/knockdown) across three cell types:

- **LDA shift**: change in mean LDA score of sALS cells, normalized by the
  baseline sALS to PN gap. Captures global, linear movement along the disease axis.
- **KNN shift**: change in the fraction of sALS cells' K=20 nearest neighbors
  that are PN cells. Captures local, non-linear movement toward PN regions.

Each metric is summarized per (gene, direction) as a signal-to-noise ratio:

    SNR = mean shift across cell types / std shift across cell types

This rewards effects that are both large and consistent across cell types. A high
mean with high variance (signal only in one cell type) scores lower than a
moderate mean with low variance. The two SNRs are z-scored so neither metric
dominates due to scale differences. Each z-SNR is then weighted by its
per-metric reliability (see below) before the two scores are averaged.

## Reliability weighting

Task 3 computes a per-metric reliability score (0-3) for each gene based on three
binary checks applied independently to LDA and KNN:

| Check | What it tests |
|---|---|
| KU/KD opposite | KU and KD produce opposite-sign shifts (gene-dose coherence) |
| KU celltype agree | KU shift has the same sign in all three cell types |
| KU exp=obs | KU direction matches expectation from expression differences |

The LDA SNR is multiplied by LDA_reliability / 3, and the KNN SNR by
KNN_reliability / 3, before they are averaged into the composite. This means each
metric is down-weighted independently: a gene that passes no LDA checks but all
KNN checks scores entirely on the KNN side, and vice versa.

## Caveat

All perturbation effects are ≤ 3% of the sALS to PN LDA gap. The ranking below
is exploratory: it represents the best available ordering given the noise floor,
not a confident prediction of rescue efficacy. The null result most likely reflects
that ALS transcriptional divergence is a network-level phenomenon that cannot be
detected through single-gene rank perturbations. See Task 3 for a full discussion.

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import zscore

RESULTS_DIR = pathlib.Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Load pre-computed metric DataFrames and reliability table from Task 3
lda_df = pd.read_csv("../data/lda_df.csv")
knn_df = pd.read_csv("../data/knn_df.csv")
reliability_df = pd.read_csv("../data/reliability_df.csv")

In [ ]:
# Compute per-metric reliability scores (0-3 each), then weight each SNR by its
# own metric-specific reliability before merging into the composite.
lda_rel_cols = ["LDA KU/KD opp", "LDA KU ct agree", "LDA KU exp=obs"]
knn_rel_cols = ["KNN KU/KD opp", "KNN KU ct agree", "KNN KU exp=obs"]

rel = reliability_df[["gene"] + lda_rel_cols + knn_rel_cols].copy()
rel["lda_reliability"] = rel[lda_rel_cols].sum(axis=1)
rel["knn_reliability"] = rel[knn_rel_cols].sum(axis=1)

merged = lda_df.merge(knn_df, on=["gene", "direction", "celltype"])
eps = 1e-6

agg = (
    merged.groupby(["gene", "direction"])
    .agg(
        mean_lda=("lda_shift", "mean"),
        std_lda=("lda_shift", "std"),
        mean_knn=("delta_pn_frac", "mean"),
        std_knn=("delta_pn_frac", "std"),
    )
    .reset_index()
)

agg["snr_lda"] = agg["mean_lda"] / (agg["std_lda"] + eps)
agg["snr_knn"] = agg["mean_knn"] / (agg["std_knn"] + eps)
agg["z_snr_lda"] = zscore(agg["snr_lda"])
agg["z_snr_knn"] = zscore(agg["snr_knn"])

# Weight each metric by its own reliability (0-3) before averaging into composite
agg = agg.merge(rel[["gene", "lda_reliability", "knn_reliability"]], on="gene")
agg["z_snr_lda_w"] = agg["z_snr_lda"] * (agg["lda_reliability"] / 3)
agg["z_snr_knn_w"] = agg["z_snr_knn"] * (agg["knn_reliability"] / 3)
agg["composite_weighted"] = (agg["z_snr_lda_w"] + agg["z_snr_knn_w"]) / 2

# Best direction per gene, ranked by weighted composite
best = (
    agg.loc[agg.groupby("gene")["composite_weighted"].idxmax()]
    .sort_values("composite_weighted", ascending=False)
    .reset_index(drop=True)
)
best["rank"] = best.index + 1
best["direction"] = best["direction"].map({"up": "↑", "down": "↓"})

print(best[["rank", "gene", "direction", "composite_weighted", "lda_reliability", "knn_reliability", "mean_lda", "mean_knn"]].to_string(index=False))

## Plot 1: Drug target ranking

Reliability-weighted composite rescue score for each gene's best perturbation
direction. Red bars indicate a net positive score (movement toward PN across both
metrics, passing enough reliability checks); blue indicates net negative or zero.
Each bar is annotated with the per-metric reliability: LDA x/3 and KNN x/3,
showing how many of the three signal quality checks each metric passed in Task 3.

Given the null result established in Task 3, this ranking is best interpreted as
a relative ordering of exploratory hypotheses rather than a prediction of rescue
efficacy. Genes with higher reliability scores are more likely to reflect a real
(if weak) biological signal rather than noise.

In [ ]:
# Horizontal bar chart of reliability-weighted composite scores
fig, ax = plt.subplots(figsize=(8, 5))
labels = best["gene"] + " " + best["direction"]
colors = ["#d62728" if s > 0 else "#1f77b4" for s in best["composite_weighted"]]
bars = ax.barh(labels, best["composite_weighted"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.invert_yaxis()

for bar, lda_r, knn_r in zip(bars, best["lda_reliability"], best["knn_reliability"]):
    x = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    ax.text(
        x + 0.02 * (ax.get_xlim()[1] - ax.get_xlim()[0]),
        y, f"LDA {int(lda_r)}/3  KNN {int(knn_r)}/3", va="center", fontsize=8
    )

ax.set_xlabel("Reliability-weighted composite score (z-SNR_LDA × LDA_rel/3 + z-SNR_KNN × KNN_rel/3) / 2")
ax.set_title(
    "Drug target ranking: best perturbation per gene (sALS → PN direction)\n"
    "Annotation: per-metric reliability (0-3 checks each: KU/KD opposite, celltype agreement, KU matches expression)"
)
plt.tight_layout()
plt.savefig("../results/drug_target_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

## Plot 2: Metric agreement scatter

Each point is a (gene, direction) pair positioned by its z-scored LDA SNR (x)
vs. z-scored KNN SNR (y). Triangles pointing up = knockup; pointing down =
knockdown.

Points near the dashed diagonal are supported by both metrics and carry more
weight in the composite. Points far off the diagonal are driven by only one
metric, making them less reliable. The scatter across all four quadrants with no
clear diagonal clustering confirms that LDA and KNN are largely independent at
this noise level, consistent with both measuring noise rather than a true
biological signal.

In [ ]:
# Scatter of z-scored LDA SNR vs KNN SNR per (gene, direction)
fig, ax = plt.subplots(figsize=(6, 5))
for _, row in agg.iterrows():
    marker = "^" if row["direction"] == "up" else "v"
    ax.scatter(row["z_snr_lda"], row["z_snr_knn"], marker=marker, s=60, zorder=3)
    ax.annotate(
        row["gene"], (row["z_snr_lda"], row["z_snr_knn"]),
        fontsize=7, xytext=(4, 4), textcoords="offset points"
    )

lim = max(abs(agg[["z_snr_lda", "z_snr_knn"]].values.ravel())) * 1.15
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.axhline(0, color="gray", lw=0.6, linestyle=":")
ax.axvline(0, color="gray", lw=0.6, linestyle=":")
ax.plot([-lim, lim], [-lim, lim], color="gray", lw=0.8, linestyle="--", label="perfect agreement")
ax.set_xlabel("Z-score: LDA SNR (mean / std across cell types)")
ax.set_ylabel("Z-score: KNN SNR (mean / std across cell types)")
ax.set_title(
    "Metric agreement (▲=knockup, ▼=knockdown)\n"
    "Points near the diagonal are supported by both metrics"
)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig("../results/drug_target_metric_agreement.png", dpi=150, bbox_inches="tight")
plt.show()